# 06 - Count vs Learned Comparison (C4) & Vectorize Interface (Part D)
**Owner:** Rabi Sandhu

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/cleaned.csv").dropna(subset=["clean_text"])
docs, labels = df["clean_text"].tolist(), df["label"].tolist()
print(len(docs), "documents")

In [ ]:
import joblib, time
from vectorize import vectorize, METHODS
from count_vectors import sparsity
MODELS = ROOT / "models"
print("Saved files:")
for p in sorted(MODELS.iterdir()):
    print(" ", p.name, f"{p.stat().st_size/1e6:.1f} MB")

## C4. Count-based vs learned on OUR data

In [ ]:
sample = docs[:500]
rows = []
for m in METHODS:
    V = np.array([vectorize(d, m, already_clean=True) for d in sample])  # text is already cleaned
    rows.append({"method": m, "dimensions": V.shape[1],
                 "% zeros (500 docs)": round(100 * (V == 0).mean(), 2)})
pd.DataFrame(rows)

In [ ]:
unseen = "یہ بالکل نیا لفظ ہے: کرکٹستان"     # a word that does not exist
for m in METHODS:
    v = vectorize(unseen, m)
    print(f"{m:9s} non-zero values: {np.count_nonzero(v)}")

**Comparison (edit in your own words):**

| | BoW / TF-IDF | Word2Vec (CBOW, Skip-gram) | fastText |
|---|---|---|---|
| Dimensions | vocabulary size (100k+) | 100 | 100 |
| Sparsity | ~99% zeros | dense, no zeros | dense |
| Unknown words | ignored | ignored | vector from char n-grams |
| What it captures | which words appear, how often, how rare | words used in similar contexts get close vectors | same + spelling / word pieces |
| Word order | no (only with n-grams) | small window | small window |

## D. Saved models & the vectorize() interface
`src/vectorize.py` loads the saved files (no retraining) and turns a RAW document into a vector. Phase 3 calls only this function.

In [ ]:
raw = "پاکستان کرکٹ ٹیم نے سری لنکا کو 3 وکٹوں سے شکست دے دی۔"
for m in METHODS:
    t = time.time(); v = vectorize(raw, m)
    print(f"{m:9s} shape={v.shape}  first values={np.round(v[:4], 3)}  ({1000*(time.time()-t):.0f} ms)")